# Channel Management — Simplified (pyvium ≥ 0.2.0)

This notebook replaces the workaround in `ChannelManagement.ipynb` using the new method introduced in **pyvium 0.2.0**.

---

## New method: `Pyvium.select_serial_number(serial_number: str)`

**Core equivalent:** `Core.IV_SelectSn(str)`  
**DLL version introduced:** 4.1239

Selects a specific device by serial number from the IviumSoft dropdown list, making it ready to connect.  
Returns the position index of the serial number in the list (0-based).

| Return value | Meaning |
|---|---|
| `0`, `1`, `2`, ... | Position of the serial number in the dropdown list |
| `DeviceNotConnectedToIviumSoftError` | Serial number not found, or a device is already connected on this channel |

### How it solves the old workaround

Previously, connecting a **specific device** to a **specific channel** required:
1. Reading all channel statuses
2. Connecting devices sequentially until reaching the target channel
3. Disconnecting all unwanted connections

With `select_serial_number`, the workflow is simply:
```python
Pyvium.select_channel(channel)          # activate the target channel tab
Pyvium.select_serial_number(serial_number)  # pre-select the device
Pyvium.connect_device()                 # connect it
```

In [ ]:
from pyvium import Pyvium, Core

Pyvium.open_driver()

## Utility helpers

These functions are unchanged from the previous notebook — they remain useful for inspecting and bulk-managing channels.

In [ ]:
def get_channels_statuses(number_of_channels: int) -> list:
    """Returns [[channel, serial_number, status_code], ...] for each channel.
    status codes: -1=no IviumSoft, 0=not connected, 1=idle, 2=busy, 3=no device
    """
    statuses = []
    for ch in range(1, number_of_channels + 1):
        Core.IV_SelectChannel(ch)
        status_code = Core.IV_getdevicestatus()
        sn = Core.IV_readSN()[1] if status_code not in [-1, 3] else ''
        statuses.append([ch, sn, status_code])
    return statuses


def disconnect_channel(channel: int) -> None:
    """Disconnects the device on a channel if it is idle."""
    Core.IV_SelectChannel(channel)
    if Core.IV_getdevicestatus() == 1:
        Core.IV_connect(0)

## Simplified: connect a specific device to a specific channel

`select_serial_number` replaces the sequential connect/disconnect workaround entirely.

In [ ]:
def connect_device_to_channel(serial_number: str, channel: int) -> None:
    """Connect a specific device (by serial number) to a specific channel.

    Uses Pyvium.select_serial_number() introduced in pyvium 0.2.0 (DLL 4.1239).
    Disconnects any existing idle device on the channel before connecting.
    """
    Pyvium.select_channel(channel)

    # Disconnect the current device if the channel is idle (not mid-measurement)
    if Core.IV_getdevicestatus() in [1]:
        Pyvium.disconnect_device()

    Pyvium.select_serial_number(serial_number)  # raises if SN not found
    Pyvium.connect_device()

## Example usage

In [ ]:
# Inspect all channels before making changes
print(get_channels_statuses(4))

In [ ]:
# Connect a specific device to channel 2
connect_device_to_channel('R31592', 2)

In [ ]:
# Verify the result
print(get_channels_statuses(4))

In [ ]:
Pyvium.close_driver()